In [ ]:
# Cell 2 — config actualizado para Vertex AI
import json, time, re, os
from pathlib import Path
import vertexai
from vertexai.batch_prediction import BatchPredictionJob
from google.cloud import storage

PROJECT_ID    = "gen-lang-client-0944146289"
LOCATION      = "us-central1"
BUCKET_NAME   = "clones-batch-alvaro"
MODEL         = "gemini-2.5-flash"
INPUT_FOLDER  = "./js_files"
OUTPUT_FOLDER = "./output/clones"
MAX_LINES     = 500
N_FILES       = 2500 

vertexai.init(project=PROJECT_ID, location=LOCATION)
storage_client = storage.Client(project=PROJECT_ID)

In [ ]:
#from vertexai.batch_prediction import BatchPredictionJob

#jobs = BatchPredictionJob.list()
#for job in jobs:
#    print(job.name, "|", job.state, "|", job.create_time)

#job_name = "projects/gen-lang-client-0944146289/locations/us-central1/batchPredictionJobs/5353428502534160384"

5353428502534160384 | 4 | 2026-04-18 05:21:30.599413+00:00
2731582003206946816 | 4 | 2026-04-12 18:33:49.481546+00:00


In [6]:
CLONE_PROMPTS = {
    1: """You are an expert in code clone generation.

Generate a Type I clone of the JavaScript code below.

Official Type I definition:
"Code fragments are exactly equal, except for possible variations
in whitespace and comments."

This means:
- Every identifier, variable name, function name, operator, and expression
  must remain completely unchanged
- The only permitted changes are whitespace (indentation, spacing, blank lines)
  and comments (add, remove, or modify them)
- Make these changes extensively throughout the entire file so the result
  looks visually different, but is logically and structurally identical

Output only the raw JavaScript code. No markdown, no backticks, no explanation.""",

    2: """You are an expert in code clone generation.

Generate a Type II clone of the JavaScript code below.

Official Type II definition:
"The structure of the code fragments is the same. However, identifier names,
data types, whitespace and comments are different."

This means:
- The control flow, logic, and number of statements must remain identical
- You must change: all identifier names (variables, parameters, functions),
  data types where applicable, whitespace throughout, and all comments
- No original identifier name should appear in the output
- Use semantically related but clearly different names

Make these changes extensively throughout the entire file.

Output only the raw JavaScript code. No markdown, no backticks, no explanation.""",

    3: """You are an expert in code clone generation.

Generate a Type III clone of the JavaScript code below.

Official Type III definition:
"In addition to changes in identifiers, variable names, data types and comments,
some parts of the code may be deleted, updated, or new fragments added."

This means:
- First apply all Type II changes: rename all identifiers, change data types,
  reformat whitespace, rewrite comments
- Then additionally: delete some non-essential statements, update or rewrite
  existing ones in equivalent ways, and add new code fragments such as
  helper variables, intermediate steps, guard clauses, or validations
- The overall functionality must remain recognizably similar

Make extensive changes throughout the entire file — both the Type II renaming
and the structural additions/removals should be clearly visible.

Output only the raw JavaScript code. No markdown, no backticks, no explanation.""",

    4: """You are an expert in code clone generation.

Generate a Type IV clone of the JavaScript code below.

Official Type IV definition:
"The fragments have divergent syntax but execute equivalent functionality."

This means:
- The output must produce exactly the same results as the original for any input
- The implementation must use a completely different approach and syntax
- Do not reuse any identifier, variable name, or function name from the original
- Choose a genuinely different strategy: if the original uses loops, consider
  recursion or functional methods; if it uses conditionals, consider lookup
  tables; if it is imperative, consider functional style — or vice versa
- The result should be unrecognizable as derived from the original

Output only the raw JavaScript code. No markdown, no backticks, no explanation.""",
}

In [3]:
def prepare_batch(n_files=N_FILES):
    input_path = Path(INPUT_FOLDER)
    js_files = sorted(
        [f for f in input_path.iterdir() if f.suffix == ".js"],
        key=lambda f: int(f.stem) if f.stem.isdigit() else f.stem
    )[:n_files]

    requests = []
    for js_file in js_files:
        try:
            original = js_file.read_text(encoding="utf-8")
        except UnicodeDecodeError:
            original = js_file.read_text(encoding="latin-1")
        lines = original.splitlines()
        if len(lines) > MAX_LINES:
            original = "\n".join(lines[:MAX_LINES])

        for clone_type in [1, 2, 3, 4]:
            request = {
                "custom_id": f"{js_file.stem}_clon_{clone_type}",
                "request": {
                    "model": f"models/{MODEL}",
                    "contents": [
                        {
                            "role": "user",
                            "parts": [
                                {"text": CLONE_PROMPTS[clone_type] + "\n\nCODE:\n" + original}
                            ]
                        }
                    ],
                    "generation_config": {
                        "temperature": 0.7,
                    }
                }
            }
            requests.append(request)

    jsonl_path = Path("batch_requests.jsonl")
    with open(jsonl_path, "w", encoding="utf-8") as f:
        for req in requests:
            f.write(json.dumps(req) + "\n")

    print(f"Files included : {len(js_files)}")
    print(f"Total requests : {len(requests)}")
    print(f"JSONL saved to : {jsonl_path}")
    return jsonl_path

jsonl_path = prepare_batch(n_files=N_FILES)

Files included : 2500
Total requests : 10000
JSONL saved to : batch_requests.jsonl


In [4]:
def submit_batch_vertex(jsonl_path):
    # subir JSONL al bucket
    bucket = storage_client.bucket(BUCKET_NAME)
    blob_name = f"input/{Path(jsonl_path).name}"
    blob = bucket.blob(blob_name)
    blob.upload_from_filename(jsonl_path)
    input_uri = f"gs://{BUCKET_NAME}/{blob_name}"
    print(f"Uploaded to: {input_uri}")

    # output también va al bucket
    output_uri = f"gs://{BUCKET_NAME}/output/"

    # mandar el batch
    print("Submitting batch job...")
    job = BatchPredictionJob.submit(
        source_model=MODEL,
        input_dataset=input_uri,
        output_uri_prefix=output_uri,
    )

    print(f"\nBatch submitted!")
    print(f"Job name : {job.resource_name}")
    print(f"State    : {job.state}")
    print(f"\nPuedes cerrar la laptop. Checa el status después.")
    return job.resource_name

job_name = submit_batch_vertex(jsonl_path)

Uploaded to: gs://clones-batch-alvaro/input/batch_requests.jsonl
Submitting batch job...
BatchPredictionJob created. Resource name: projects/648630313845/locations/us-central1/batchPredictionJobs/2731582003206946816
To use this BatchPredictionJob in another session:
job = batch_prediction.BatchPredictionJob('projects/648630313845/locations/us-central1/batchPredictionJobs/2731582003206946816')
View Batch Prediction Job:
https://console.cloud.google.com/ai/platform/locations/us-central1/batch-predictions/2731582003206946816?project=648630313845

Batch submitted!
Job name : projects/648630313845/locations/us-central1/batchPredictionJobs/2731582003206946816
State    : 2

Puedes cerrar la laptop. Checa el status después.


In [ ]:
# cmabiar antes de pedir resulatdos
# job_name="batches/xw0679kljjcn9uc677ti6x9j5ko732wikwmj"

In [8]:
# Cell 6 — checar status
def check_status_vertex(job_name):
    job = BatchPredictionJob(job_name)
    job.refresh()
    print(f"State    : {job.state}")
    print(f"Error    : {job.error}")
    if str(job.state) == "JobState.JOB_STATE_SUCCEEDED":
        print("\nTerminó. Corre Cell 7 para descargar.")
    else:
        print("\nSigue procesando. Checa en unos minutos.")
    return job

job = check_status_vertex(job_name)

State    : 4
Error    : 

Sigue procesando. Checa en unos minutos.


In [ ]:
# Solo para cancelar trabajo
#client.batches.cancel(name=job_name)

In [13]:
def strip_markdown(code):
    import re
    code = re.sub(r"^```[a-zA-Z]*\n?", "", code.strip())
    code = re.sub(r"\n?```$", "", code.strip())
    return code.strip()

def download_and_save_vertex(job_name):
    job = BatchPredictionJob(job_name)
    job.refresh()

    state = str(job.state)
    print(f"State: {state} ({job.state})")

    # acepta tanto el nombre como el número
    if "SUCCEEDED" not in state and job.state != 4:
        print(f"No terminó aún: {job.state}")
        return

    # buscar archivos de output en el bucket
    bucket = storage_client.bucket(BUCKET_NAME)
    blobs = list(bucket.list_blobs(prefix="output/"))

    out_path = Path(OUTPUT_FOLDER)
    out_path.mkdir(parents=True, exist_ok=True)

    success, failed = 0, 0
    for blob in blobs:
        if not blob.name.endswith(".jsonl"):
            continue
        content = blob.download_as_text()
        for line in content.strip().split("\n"):
            if not line:
                continue
            try:
                data = json.loads(line)
                custom_id = data.get("custom_id", "unknown")
                filename  = f"{custom_id}.js"
                text = data["response"]["candidates"][0]["content"]["parts"][0]["text"]
                code = strip_markdown(text)
                (out_path / filename).write_text(code, encoding="utf-8")
                print(f"  saved: {filename}")
                success += 1
            except Exception as e:
                print(f"  failed: {e}")
                failed += 1

    print(f"\nSaved  : {success} clones -> {OUTPUT_FOLDER}")
    print(f"Failed : {failed}")

download_and_save_vertex(job_name)

State: 4 (4)
  saved: 270_clon_3.js
  saved: 1871_clon_1.js
  saved: 362_clon_2.js
  saved: 1736_clon_3.js
  saved: 147_clon_2.js
  saved: 1504_clon_1.js
  saved: 2703_clon_3.js
  saved: 164_clon_1.js
  saved: 2757_clon_3.js
  saved: 1597_clon_1.js
  saved: 1587_clon_1.js
  saved: 4888_clon_4.js
  saved: 2232_clon_2.js
  saved: 1920_clon_4.js
  saved: 2294_clon_1.js
  saved: 1841_clon_2.js
  saved: 2436_clon_3.js
  saved: 160_clon_4.js
  saved: 3685_clon_4.js
  saved: 287_clon_1.js
  saved: 238_clon_3.js
  saved: 3351_clon_3.js
  saved: 3220_clon_1.js
  saved: 187_clon_2.js
  saved: 1779_clon_1.js
  saved: 3168_clon_2.js
  saved: 4179_clon_4.js
  saved: 4198_clon_1.js
  saved: 408_clon_3.js
  saved: 856_clon_3.js
  saved: 3880_clon_1.js
  saved: 652_clon_3.js
  saved: 1813_clon_2.js
  saved: 1162_clon_3.js
  saved: 4850_clon_4.js
  saved: 2249_clon_2.js
  saved: 4007_clon_2.js
  saved: 683_clon_2.js
  saved: 4142_clon_1.js
  saved: 2989_clon_2.js
  saved: 2688_clon_2.js
  saved: 3331_c

In [12]:
job = client.batches.get(name=job_name)
inicio = job.create_time
fin    = job.end_time
duracion = fin - inicio
print(f"Inicio   : {inicio}")
print(f"Fin      : {fin}")
print(f"Duracion : {duracion}")

Inicio   : 2026-04-12 08:30:34.089185+00:00
Fin      : 2026-04-12 09:25:42.369379+00:00
Duracion : 0:55:08.280194


In [33]:
# lista todos tus batches
for batch in client.batches.list():
    print(f"{batch.name} — {batch.state.value}")

batches/xw0679kljjcn9uc677ti6x9j5ko732wikwmj — JOB_STATE_SUCCEEDED
batches/g4uasin3i6ibrp0zy5eccg3ub4n8z93vah1g — JOB_STATE_CANCELLED
